<a href="https://colab.research.google.com/github/Hwk040319/MJY-ML/blob/main/00_quickstart_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 배터리 열폭주 이미지 분류 · Colab 퀵스타트

## 0. GPU 확인

In [ ]:
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
# False 면 런타임 -> 런타임 유형 변경 -> T4 GPU 선택 후 이 셀 다시 실행

## 1. 코드 내려받기

In [ ]:
!git clone https://github.com/Hwk040319/MJY-ML.git
%cd MJY-ML
!pip install -q -r requirements.txt

## 2. 데이터 내려받기



In [ ]:
!pip install -q gdown

# 1회차 강의 실습용 (약 700장, 22MB)
FILE_ID = '13DWY5tg_L4SYxujkdVQ89qEQZC08lr7L'

# 회차 사이 팀 실험용 (전체, 11GB) — 위 줄을 주석 처리하고 아래를 사용
# FILE_ID = '1PJNyDDdYd47wXD83DiW9PqFzbe7TLl0n'

!gdown "https://drive.google.com/uc?id=$FILE_ID" -O data.tar
!mkdir -p data && tar -xf data.tar -C data
!ls data

## 3. 데이터 검사




In [ ]:
!python check_data.py --data-root data

## 3.5 데이터 구조와 전처리 직접 확인 (선택)

채점이나 제출과 무관한 확인용 셀입니다. `train_baseline.py`를 실행하기 전에, 실제로 어떤 이미지가 어떤 라벨로 들어가는지, 그리고 슬라이드 13에서 설명한 전처리 5단계가 코드에서 정확히 어떻게 일어나는지 직접 눈으로 확인합니다.

In [ ]:
# labels.csv 구조 확인 — 이미지 파일명과 라벨이 어떻게 짝지어져 있는지
import pandas as pd

df = pd.read_csv('data/train/labels.csv')
print('열 구성:', list(df.columns))          # image_name, target, original_stage, experiment_id
print('행 개수:', len(df))
df.head()

In [ ]:
# 클래스(초기/중기/후기)별로 실제 이미지를 한 장씩 눈으로 확인
from PIL import Image
import matplotlib.pyplot as plt

CLASS_NAMES = ['초기', '중기', '후기']
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for cls in range(3):
    row = df[df['target'] == cls].iloc[0]                       # 해당 클래스의 첫 번째 행
    img = Image.open(f"data/train/images/{row['image_name']}")  # 파일명으로 실제 이미지 열기
    axes[cls].imshow(img)
    axes[cls].set_title(f"{CLASS_NAMES[cls]} · {row['image_name']}")
    axes[cls].axis('off')
plt.suptitle('클래스별 샘플 이미지 (원본, 전처리 전)')
plt.show()

print('원본 이미지 크기(W,H):', img.size)   # 실험마다 원본 해상도가 다를 수 있습니다

In [ ]:
# 슬라이드 13의 전처리 흐름(Resize -> ToTensor -> Normalize)을 한 단계씩 직접 실행
from torchvision import transforms
from common import IMAGENET_MEAN, IMAGENET_STD, get_transforms

sample_name = df.iloc[0]['image_name']
sample_img = Image.open(f"data/train/images/{sample_name}").convert('RGB')
print('0) 원본 크기:', sample_img.size)

step1 = transforms.Resize((224, 224))(sample_img)
print('1) Resize 후 크기:', step1.size)                      # (224, 224)로 통일

step2 = transforms.ToTensor()(step1)
print('2) ToTensor 후 shape:', tuple(step2.shape),
      '값 범위:', round(step2.min().item(), 3), '~', round(step2.max().item(), 3))
# [3, 224, 224], 0~255 픽셀값을 255로 나눠 0~1 범위로 변환

step3 = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)(step2)
print('3) Normalize 후 값 범위:', round(step3.min().item(), 2), '~', round(step3.max().item(), 2))
# ImageNet 평균/표준편차로 재조정 -> 사전학습된 ResNet18이 기대하는 입력 분포에 맞춤

final = get_transforms(224, train=False)(sample_img)
print('get_transforms() 결과와 동일한가:', torch.allclose(final, step3))  # True면 위 세 단계와 같은 전처리

## 4. Baseline 학습

In [ ]:
!python train_baseline.py \
  --data-root data \
  --output-dir outputs/baseline \
  --epochs 5 \
  --batch-size 32 \
  --lr 1e-3

## 5. 결과 확인

In [ ]:
import json
with open('outputs/baseline/validation_report.json', encoding='utf-8') as f:
    report = json.load(f)
print('Macro F1:', round(report['macro_f1'], 4))
print('Accuracy:', round(report['accuracy'], 4))
print('Class F1 (초기/중기/후기):', [round(v, 4) for v in report['class_f1']])

# 학습이 진행되며 loss와 validation 점수가 어떻게 변했는지 바로 확인
from IPython.display import Image as DisplayImage, display
display(DisplayImage(filename='outputs/baseline/learning_curves.png'))
display(DisplayImage(filename='outputs/baseline/confusion_matrix.png'))

## 5.5 이미지 한 장 예측 (선택)

In [ ]:
from pathlib import Path
sample_image = next(Path('data/public_val/images').glob('*.png'))
print('예측할 이미지:', sample_image)
!python predict_one.py --image "$sample_image" --checkpoint outputs/baseline/best_model.pt

## 6. 개선 실험 (회차 사이)

In [ ]:
# A. 데이터 증강
!python train_baseline.py --data-root data --augment --epochs 5 --output-dir outputs/exp_aug

# B. 클래스 가중치
!python train_baseline.py --data-root data --use-class-weights --epochs 5 --output-dir outputs/exp_weight

# C. 전체 미세조정 (작은 learning rate 필수)
!python train_baseline.py --data-root data --unfreeze --lr 1e-4 --epochs 5 --output-dir outputs/exp_ft

# D. 옵티마이저 변경 (adamw(기본)/adam/sgd 중 선택)
!python train_baseline.py --data-root data --optimizer sgd --epochs 5 --output-dir outputs/exp_sgd

## 7. 최종 제출 · 2회차 전날 14:00 마감

In [ ]:
import torch
FINAL = 'outputs/exp_aug/best_model.pt'   # 최종 선택한 경로로 변경
ckpt = torch.load(FINAL, map_location='cpu', weights_only=False)
print('Public Val Macro F1:', round(ckpt['macro_f1'], 4), '| epoch:', ckpt['epoch'])

from google.colab import files
files.download(FINAL)
# [팀명]_best_model.pt 로 이름 변경 후 제출 폼에 업로드

## 8. (선택) CNN을 처음부터 학습시켜보고 싶다면

여기까지는 이미 학습된 ResNet18을 **빌려 쓰는** 전이학습이었습니다. "CNN 내부가 정확히 어떻게 학습되는가"를 직접 만져보고 싶다면, 같은 저장소의 [`02_cnn_theory_mnist.ipynb`](https://colab.research.google.com/github/Hwk040319/MJY-ML/blob/main/02_cnn_theory_mnist.ipynb)를 열어보세요. MNIST 손글씨 숫자로 CNN을 처음부터 학습시키면서 epoch·optimizer·activation을 직접 바꿔보는 선택 실습입니다. 채점과는 무관합니다.